# 03 — LoRA Fine-Tuning with PEFT

**Project:** TalentMatch.ai — Fine-tuned DistilBERT for Resume-JD Scoring  
**Goal of this notebook:** Fine-tune the same DistilBERT model using LoRA (Low-Rank Adaptation) — a parameter-efficient method that trains only ~1% of the model's weights. We then compare results against Phase 2's full fine-tuning.

---

## What is LoRA?
LoRA (Low-Rank Adaptation) is a technique that freezes all original model weights and injects small trainable matrices into specific layers. Instead of updating all 66M parameters, LoRA adds pairs of small matrices (A and B) to the attention layers. Only these matrices are trained — everything else stays frozen.

The math: instead of updating a weight matrix W directly, LoRA approximates the update as W + A×B, where A and B are much smaller matrices. The rank `r` controls how small they are. With `r=8`, we're saying the weight update can be approximated by two matrices of rank 8 — a massive compression.

**Why this matters for your portfolio:**  
LoRA is how the industry fine-tunes large models cheaply. GPT-3, LLaMA, and most production fine-tuned models use parameter-efficient methods like LoRA. Knowing this pattern signals seniority.

## What is PEFT?
PEFT (Parameter-Efficient Fine-Tuning) is HuggingFace's library that implements LoRA and other efficient fine-tuning methods. We wrap our model with a `LoraConfig` and it automatically injects the trainable matrices.

## Notebook Flow
1. Install dependencies
2. Set up Colab Secrets for HuggingFace token
3. Mount Drive and load clean CSVs
4. Load tokenizer and build PyTorch Dataset
5. Load DistilBERT and apply LoRA config
6. Configure TrainingArguments
7. Train with Trainer
8. Evaluate on test set
9. Push LoRA adapter to HuggingFace Hub
10. Inference sanity check
11. Full benchmark comparison: Full Fine-Tuning vs LoRA


## ⚠️ Before Running: Set Up Colab Secrets

**Never hardcode your HuggingFace token in the notebook.** Use Colab Secrets instead:

1. Click the 🔑 **key icon** in the left sidebar in Colab
2. Click **Add new secret**
3. Name: `HF_TOKEN`
4. Value: paste your HuggingFace write token from `huggingface.co/settings/tokens`
5. Toggle **Notebook access** to ON

The token will be available via `userdata.get('HF_TOKEN')` and will never appear in the notebook file or git history.

Also make sure your runtime is set to **T4 GPU**: Runtime → Change runtime type → T4 GPU


In [ ]:
# =============================================================
# Step 1 — Install dependencies
# =============================================================
# peft: HuggingFace's Parameter-Efficient Fine-Tuning library
#       provides LoRA, prefix tuning, prompt tuning, and more
# All other packages are the same as notebook 02.
# We let Colab use its pre-installed versions to avoid conflicts.
# =============================================================

!pip install -q -U transformers accelerate huggingface_hub evaluate peft

In [ ]:
# =============================================================
# Step 2 — Authenticate with HuggingFace using Colab Secrets
# =============================================================
# We use Colab's userdata system to retrieve the token securely.
# The token is stored in Colab's secret manager (the key icon
# in the left sidebar) and never written to the notebook file.
#
# This is the correct pattern for credentials in notebooks:
# - NOT: login(token="hf_abc123...")  ← exposes token in file
# - YES: login(token=userdata.get('HF_TOKEN'))  ← secure
# =============================================================

from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Authenticated with HuggingFace Hub.")

In [ ]:
# =============================================================
# Step 3 — Mount Drive and load clean CSVs
# =============================================================
# Same as notebook 02 — load the preprocessed splits from Drive.
# No need to re-run Phase 1 preprocessing.
# =============================================================

from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

data_path = "/content/drive/MyDrive/TalentMatch-AI/"

train_df = pd.read_csv(f"{data_path}train.csv")
val_df   = pd.read_csv(f"{data_path}val.csv")
test_df  = pd.read_csv(f"{data_path}test.csv")

print(f"Train:      {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test:       {len(test_df)} samples")
print(f"Score range: {train_df['score'].min():.2f} - {train_df['score'].max():.2f}")

In [ ]:
# =============================================================
# Step 4 — Load tokenizer and build PyTorch Dataset
# =============================================================
# Identical to notebook 02. We tokenize resume-JD pairs as
# sentence pairs: [CLS] resume [SEP] jd [SEP]
# Labels are normalized to 0-1 for training stability.
# =============================================================

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ResumeJDDataset(Dataset):
    """
    PyTorch Dataset for resume-JD pairs.
    Returns tokenized inputs and normalized float labels (score/100).
    """
    def __init__(self, df, tokenizer):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoding = self.tokenizer(
            row['resume'],
            row['jd'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(row['score'] / 100.0, dtype=torch.float)
        }

train_dataset = ResumeJDDataset(train_df, tokenizer)
val_dataset   = ResumeJDDataset(val_df,   tokenizer)
test_dataset  = ResumeJDDataset(test_df,  tokenizer)

print(f"Datasets built: {len(train_dataset)} train | {len(val_dataset)} val | {len(test_dataset)} test")

In [ ]:
# =============================================================
# Step 5 — Load DistilBERT and apply LoRA configuration
# =============================================================
# This is the key difference from notebook 02.
#
# After loading the base model, we wrap it with get_peft_model()
# which injects LoRA matrices and FREEZES all original weights.
#
# LoraConfig parameters explained:
#
# task_type=SEQ_CLS:
#   Tells PEFT this is a sequence classification/regression task.
#   Determines which layers get the LoRA adapter applied.
#
# r=8:
#   The rank of the LoRA matrices. Lower = fewer parameters.
#   r=8 means each weight update is approximated by two matrices
#   of shape (768x8) and (8x768) instead of one (768x768) matrix.
#   This compresses the update from 589,824 params to 12,288.
#
# lora_alpha=16:
#   Scaling factor for the LoRA update. The actual update applied
#   is (alpha/r) * A*B = (16/8) * A*B = 2 * A*B.
#   Higher alpha = LoRA has more influence relative to frozen weights.
#
# lora_dropout=0.1:
#   Dropout applied to LoRA layers during training to prevent
#   overfitting in the small adapter matrices.
#
# target_modules=['q_lin', 'v_lin']:
#   Which attention layers to inject LoRA into.
#   q_lin = query projection, v_lin = value projection.
#   These are the most impactful layers for task adaptation.
#   k_lin (key) is typically left frozen as it contributes less.
#
# bias='none':
#   Don't train bias terms — keeps parameter count minimal.
# =============================================================

from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

# Load base model with regression head
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
)

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q_lin', 'v_lin'],
    bias='none',
)

# Wrap model with LoRA — this freezes base weights and adds adapters
model = get_peft_model(base_model, lora_config)

# Compare trainable vs total parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable percentage: {100 * trainable / total:.2f}%")
print()
model.print_trainable_parameters()

In [ ]:
# =============================================================
# Step 6 — Define metrics and configure TrainingArguments
# =============================================================
# Same metrics as notebook 02: MAE and RMSE on 0-100 scale.
#
# TrainingArguments are identical to notebook 02 with one change:
# output_dir points to a different checkpoint folder so we don't
# overwrite the full fine-tuning checkpoints from Phase 2.
#
# Note: LoRA trains fewer parameters so it typically converges
# faster. We keep 4 epochs to make the comparison fair.
# =============================================================

import numpy as np
from transformers import TrainingArguments

def compute_metrics(eval_pred):
    """
    MAE and RMSE on the 0-100 scale.
    Trainer passes predictions and labels in 0-1 scale.
    """
    predictions, labels = eval_pred
    predictions = predictions.flatten() * 100
    labels      = labels.flatten() * 100
    mae  = np.mean(np.abs(predictions - labels))
    rmse = np.sqrt(np.mean((predictions - labels) ** 2))
    return {
        'mae':  round(float(mae),  4),
        'rmse': round(float(rmse), 4),
    }

OUTPUT_DIR = "/content/drive/MyDrive/TalentMatch-AI/checkpoints/lora_finetune"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='mae',
    greater_is_better=False,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
)

print("Metrics and TrainingArguments configured.")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================
# Step 7 — Train with Trainer
# =============================================================
# The training loop is identical to notebook 02.
# The key difference is invisible: only the LoRA adapter weights
# (q_lin and v_lin matrices) are being updated. All 66M base
# DistilBERT weights are frozen and unchanged.
#
# Watch the training time vs notebook 02 — it should be similar
# or slightly faster despite fewer parameters, because the
# forward pass still processes the full model. The speedup from
# LoRA is mainly in memory usage, not raw training speed.
#
# Expected training time: ~6-10 minutes on T4 GPU
# =============================================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
print(f"Training samples: {len(train_dataset)}")
print(f"Epochs:           {training_args.num_train_epochs}")
print(f"Batch size:       {training_args.per_device_train_batch_size}")
print(f"Steps per epoch:  {len(train_dataset) // training_args.per_device_train_batch_size}")
print("---")

train_result = trainer.train()

print("\nTraining complete!")
print(f"Total steps:   {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")

In [ ]:
# =============================================================
# Step 8 — Evaluate on test set
# =============================================================
# Same evaluation as notebook 02.
# We compare against both the naive baseline AND the full
# fine-tuning results from Phase 2 (MAE 11.95, RMSE 14.90).
#
# Key question: does LoRA with ~1% of the parameters achieve
# comparable accuracy to full fine-tuning with 100%?
# =============================================================

print("Evaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)

# Naive baseline
mean_train_score = train_df['score'].mean()
baseline_mae  = np.mean(np.abs(test_df['score'] - mean_train_score))
baseline_rmse = np.sqrt(np.mean((test_df['score'] - mean_train_score) ** 2))

print("\n===== BENCHMARK REPORT =====")
print(f"{'Metric':<20} {'Baseline':>12} {'Full FT (Ph2)':>14} {'LoRA (Ph3)':>12}")
print("-" * 60)
print(f"{'MAE':<20} {baseline_mae:>12.4f} {'11.9511':>14} {test_results['eval_mae']:>12.4f}")
print(f"{'RMSE':<20} {baseline_rmse:>12.4f} {'14.9036':>14} {test_results['eval_rmse']:>12.4f}")
print(f"{'Params Trained':<20} {'N/A':>12} {'66.9M (100%)':>14} {'~0.3M (~0.5%)':>12}")
print("=" * 60)

lora_mae  = test_results['eval_mae']
full_mae  = 11.9511
diff      = lora_mae - full_mae
direction = "better" if diff < 0 else "worse"
print(f"\nLoRA is {abs(diff):.4f} MAE points {direction} than full fine-tuning.")

In [ ]:
# =============================================================
# Step 9 — Push LoRA adapter to HuggingFace Hub
# =============================================================
# Important distinction from notebook 02:
# With LoRA we push only the ADAPTER weights — not the full model.
# The adapter is tiny (~3MB) compared to the full model (~268MB).
#
# When loading the model later, HuggingFace automatically merges
# the adapter with the base DistilBERT weights on the fly.
#
# The repo name uses '-lora' suffix to distinguish it from
# the full fine-tuned model pushed in Phase 2.
#
# Token is loaded securely from Colab Secrets — never hardcoded.
# =============================================================

HF_REPO = "LucasLisboadev/TalentMatch-AI-lora"

print(f"Pushing LoRA adapter to: https://huggingface.co/{HF_REPO}")

# Save locally first to ensure complete upload
model.save_pretrained("/content/talentmatch_lora")
tokenizer.save_pretrained("/content/talentmatch_lora")

# Push to Hub
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"\nLoRA adapter pushed successfully!")
print(f"View at: https://huggingface.co/{HF_REPO}")

In [ ]:
# =============================================================
# Step 10 — Inference sanity check
# =============================================================
# We run the same two test pairs as notebook 02:
#   1. Strong match — Python engineer → Python backend role
#   2. Weak match   — Nurse → Python backend role
#
# Loading a PEFT model from Hub requires PeftModel.from_pretrained()
# instead of AutoModel.from_pretrained(). It loads the base model
# first then applies the adapter on top.
#
# We also use longer, more realistic test pairs that better match
# the training distribution — this should give more reliable results
# than the short synthetic pairs from notebook 02.
# =============================================================

from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

def predict_score(resume, jd, tokenizer, model):
    """Run inference on a resume-JD pair. Returns score 0-100."""
    inputs = tokenizer(
        resume, jd,
        max_length=512,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )
    with torch.no_grad():
        output = model(**inputs)
    score = output.logits.squeeze().item() * 100
    return round(float(np.clip(score, 0, 100)), 2)

# Load LoRA model from Hub
inf_tokenizer  = AutoTokenizer.from_pretrained("distilbert-base-uncased")
inf_base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=1
)
inf_model = PeftModel.from_pretrained(inf_base_model, HF_REPO)
inf_model.eval()

# Strong match — longer, more realistic pair matching training distribution
strong_resume = """Senior Backend Engineer with 6 years of experience designing and building
scalable REST APIs using Python and FastAPI. Proficient in PostgreSQL, Redis, and Docker.
Led a team of 4 engineers to deliver a microservices architecture serving 2M daily users.
Strong experience with CI/CD pipelines, AWS deployments, and agile development practices."""

strong_jd = """We are looking for a Backend Engineer with 4+ years of Python experience.
The ideal candidate has hands-on expertise with FastAPI or Django, PostgreSQL, and Docker.
Experience leading small engineering teams is a strong plus. You will design and maintain
REST APIs, collaborate with frontend teams, and own backend architecture decisions."""

# Weak match — completely unrelated field
weak_resume = """Registered Nurse with 8 years of ICU experience in high-pressure clinical environments.
Skilled in patient care, medication administration, IV therapy, and clinical documentation.
Proficient in Epic EHR system. Certified in ACLS and BLS. Strong communication and
teamwork skills developed through cross-functional collaboration with physicians and staff."""

weak_jd = """We are looking for a Backend Engineer with 4+ years of Python experience.
The ideal candidate has hands-on expertise with FastAPI or Django, PostgreSQL, and Docker.
Experience leading small engineering teams is a strong plus. You will design and maintain
REST APIs, collaborate with frontend teams, and own backend architecture decisions."""

strong_score = predict_score(strong_resume, strong_jd, inf_tokenizer, inf_model)
weak_score   = predict_score(weak_resume,   weak_jd,   inf_tokenizer, inf_model)

print("===== INFERENCE SANITY CHECK (LoRA) =====")
print(f"Strong match score: {strong_score}/100")
print(f"Weak match score:   {weak_score}/100")
print(f"Difference:         {strong_score - weak_score:.2f} points")
print()
if strong_score > weak_score:
    print("✓ Model correctly scores strong match higher than weak match.")
else:
    print("✗ Scores inverted — model may need more training data to generalize.")

In [ ]:
# =============================================================
# Step 11 — Final benchmark comparison
# =============================================================
# Side-by-side summary of Phase 2 (full fine-tuning) vs
# Phase 3 (LoRA). This goes directly into the README.
#
# The key insight: LoRA achieves comparable accuracy using only
# a fraction of the trainable parameters. This is the core
# value proposition of parameter-efficient fine-tuning.
# =============================================================

print("===== TALENTMATCH.AI — FINAL BENCHMARK REPORT =====")
print()
print(f"{'Metric':<25} {'Baseline':>10} {'Full FT':>10} {'LoRA':>10}")
print("-" * 57)
print(f"{'Test MAE (↓ better)':<25} {baseline_mae:>10.2f} {'11.95':>10} {test_results['eval_mae']:>10.4f}")
print(f"{'Test RMSE (↓ better)':<25} {baseline_rmse:>10.2f} {'14.90':>10} {test_results['eval_rmse']:>10.4f}")
print(f"{'Trainable Params':<25} {'N/A':>10} {'66.9M':>10} {'~0.3M':>10}")
print(f"{'% of Model Trained':<25} {'N/A':>10} {'100%':>10} {'~0.5%':>10}")
print(f"{'Training Time (T4)':<25} {'N/A':>10} {'~90s':>10} {'~?s':>10}")
print(f"{'HF Hub Size':<25} {'N/A':>10} {'268MB':>10} {'~3MB':>10}")
print("=" * 57)
print()
print("Update the README benchmark table with these numbers.")
print(f"LoRA Hub: https://huggingface.co/{HF_REPO}")
print(f"Full FT Hub: https://huggingface.co/LucasLisboadev/TalentMatch-AI-full")

## Phase 3 Complete — LoRA Fine-Tuning Summary

| Metric | Baseline | Full Fine-Tuning | LoRA |
|---|---|---|---|
| Test MAE | 17.69 | 11.95 | TBD |
| Test RMSE | 21.23 | 14.90 | TBD |
| Trainable Params | N/A | 66.9M (100%) | ~0.3M (~0.5%) |
| HF Hub Size | N/A | 268MB | ~3MB |

*(Update with your actual LoRA results after running)*

**Next steps:**
- Update the README benchmark table with final numbers
- Build the Gradio demo (Phase 4)
- Deploy to HuggingFace Spaces
